# Modulo 4 - Analise Exploratoria de Dados Completa

### Curso Introdutorio de Python para Ciencia de Dados

**Tema:** consolidacao, qualidade dos dados, storytelling e conclusoes

**Dataset:** Brazilian Cities, com informacoes reais sobre municipios brasileiros.

---

## Checklist atendido

- Notebook de EDA estruturado
- Analise guiada por perguntas
- Storytelling com dados
- Conclusoes bem escritas

## 1. Problema analitico

A pergunta central deste notebook e:

> **Como os indicadores economicos, demograficos e territoriais ajudam a explicar diferencas de desenvolvimento humano entre os municipios brasileiros?**

A partir dela, a analise sera conduzida por quatro perguntas:

1. O dataset esta adequado para analise?
2. Como se distribuem IDHM, populacao, area, densidade e PIB per capita?
3. Existem diferencas relevantes entre regioes, capitais e municipios do interior?
4. Quais insights podem apoiar decisoes publicas, academicas ou de negocio?

## 2. Importacoes e configuracoes

In [ ]:
from pathlib import Path
import io
import warnings

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams.update({
    'figure.dpi': 115,
    'figure.facecolor': 'white',
    'axes.facecolor': '#FAFAFA',
    'axes.grid': True,
    'grid.alpha': 0.35,
})

## 3. Carregamento dos dados

O arquivo CSV do curso possui uma formatacao especifica. Por isso, a funcao abaixo faz uma leitura controlada antes de transformar os dados em `DataFrame`.

In [ ]:
def carregar_brazilian_cities(caminho):
    with open(caminho, 'r', encoding='utf-8', newline='') as arquivo:
        bruto = arquivo.read()

    linhas = bruto.split('\r\n')

    def corrigir(linha):
        if linha.startswith('"') and linha.endswith('"'):
            linha = linha[1:-1]
        return linha.replace('""', '"')

    conteudo = '\n'.join(corrigir(linha) for linha in linhas if linha.strip())
    return pd.read_csv(io.StringIO(conteudo))

candidatos = [
    Path('../../modulo-3/datasets/brazilian_city.csv'),
    Path('../../projeto-final/datasets/brazilian_city.csv'),
    Path('../datasets/brazilian_city.csv'),
]

caminho_dataset = next(caminho for caminho in candidatos if caminho.exists())
df_original = carregar_brazilian_cities(caminho_dataset)

print(f'Dataset carregado de: {caminho_dataset}')
print(f'Linhas: {df_original.shape[0]:,} | Colunas: {df_original.shape[1]}')
df_original.head()

## 4. Preparacao e enriquecimento

Nesta etapa sao criadas colunas auxiliares para facilitar a analise: regiao, populacao final, densidade demografica, tipo de municipio e faixa de IDHM.

In [ ]:
df = df_original.copy()

df = df.rename(columns={
    'IDHM Ranking 2010': 'IDHM_Ranking',
    'IBGE_CROP_PRODUCTION_$': 'IBGE_CROP_PROD',
    'WAL-MART': 'WALMART',
    'IBGE_1-4': 'IBGE_1a4',
    'IBGE_5-9': 'IBGE_5a9',
    'IBGE_10-14': 'IBGE_10a14',
    'IBGE_15-59': 'IBGE_15a59',
    'IBGE_60+': 'IBGE_60mais',
})

regioes = {
    'Norte': ['AC', 'AP', 'AM', 'PA', 'RO', 'RR', 'TO'],
    'Nordeste': ['AL', 'BA', 'CE', 'MA', 'PB', 'PE', 'PI', 'RN', 'SE'],
    'Centro-Oeste': ['DF', 'GO', 'MT', 'MS'],
    'Sudeste': ['ES', 'MG', 'RJ', 'SP'],
    'Sul': ['PR', 'RS', 'SC'],
}

df['REGIAO'] = df['STATE'].map({uf: regiao for regiao, ufs in regioes.items() for uf in ufs})
df['POP_FINAL'] = df['IBGE_RES_POP'].where(df['IBGE_RES_POP'] > 0, df['ESTIMATED_POP'])
df['DENSIDADE'] = np.where(df['AREA'] > 0, df['POP_FINAL'] / df['AREA'], np.nan)
df['TIPO'] = df['CAPITAL'].map({1: 'Capital', 0: 'Interior'})
df['IDHM_FAIXA'] = pd.cut(
    df['IDHM'],
    bins=[0, 0.499, 0.599, 0.699, 0.799, 1],
    labels=['Muito baixo', 'Baixo', 'Medio', 'Alto', 'Muito alto'],
)

df[['CITY', 'STATE', 'REGIAO', 'POP_FINAL', 'DENSIDADE', 'GDP_CAPITA', 'IDHM', 'IDHM_FAIXA']].head()

## 5. Qualidade dos dados

Antes de tirar conclusoes, e necessario verificar tipos, nulos, duplicidades e valores extremos.

In [ ]:
qualidade = pd.DataFrame({
    'coluna': df.columns,
    'tipo': df.dtypes.astype(str).values,
    'nulos': df.isna().sum().values,
    'nulos_%': (df.isna().mean().values * 100).round(2),
    'unicos': df.nunique(dropna=True).values,
})

qualidade.sort_values('nulos_%', ascending=False).head(20)

In [ ]:
duplicados = df.duplicated(subset=['CITY', 'STATE']).sum()
print(f'Duplicidades por cidade/estado: {duplicados}')

variaveis_chave = ['IDHM', 'GDP_CAPITA', 'POP_FINAL', 'AREA', 'DENSIDADE']
df[variaveis_chave].describe().T.round(2)

**Leitura da qualidade:** o conjunto tem boa cobertura para as variaveis centrais da analise. Como populacao, area e PIB per capita possuem grande variacao, a analise deve usar mediana, quartis e escala logaritmica quando necessario.

## 6. Analise guiada

### 6.1 Como o IDHM se distribui?

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(df['IDHM'].dropna(), bins=35, kde=True, color='#4C72B0', ax=ax)
ax.axvline(df['IDHM'].mean(), color='#C44E52', linestyle='--', linewidth=2, label=f"Media: {df['IDHM'].mean():.3f}")
ax.axvline(df['IDHM'].median(), color='#55A868', linestyle='-.', linewidth=2, label=f"Mediana: {df['IDHM'].median():.3f}")
ax.set_title('Distribuicao do IDHM nos municipios brasileiros')
ax.set_xlabel('IDHM')
ax.set_ylabel('Quantidade de municipios')
ax.legend()
plt.tight_layout()
plt.show()

df['IDHM_FAIXA'].value_counts().sort_index()

**Insight:** a distribuicao do IDHM permite observar a concentracao dos municipios em faixas intermediarias e identificar grupos que precisam de maior atencao.

### 6.2 Quais regioes apresentam melhores indicadores?

In [ ]:
resumo_regiao = df.groupby('REGIAO').agg(
    municipios=('CITY', 'count'),
    idhm_mediano=('IDHM', 'median'),
    idhm_medio=('IDHM', 'mean'),
    pib_pc_mediano=('GDP_CAPITA', 'median'),
    densidade_mediana=('DENSIDADE', 'median'),
    populacao_total=('POP_FINAL', 'sum'),
).sort_values('idhm_mediano', ascending=False)

resumo_regiao.round(2)

In [ ]:
ordem_regioes = resumo_regiao.index.tolist()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=df, x='REGIAO', y='IDHM', order=ordem_regioes, ax=axes[0])
axes[0].set_title('IDHM por regiao')
axes[0].set_xlabel('Regiao')
axes[0].set_ylabel('IDHM')
axes[0].tick_params(axis='x', rotation=20)

sns.barplot(data=resumo_regiao.reset_index(), x='REGIAO', y='pib_pc_mediano', order=ordem_regioes, ax=axes[1])
axes[1].set_title('PIB per capita mediano por regiao')
axes[1].set_xlabel('Regiao')
axes[1].set_ylabel('PIB per capita mediano')
axes[1].tick_params(axis='x', rotation=20)
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda valor, _: f'R$ {valor:,.0f}'))

plt.tight_layout()
plt.show()

**Insight:** as regioes nao diferem apenas pela media; a dispersao tambem importa. Boxplots ajudam a perceber desigualdades internas dentro da mesma regiao.

### 6.3 Capitais se comportam de forma diferente dos municipios do interior?

In [ ]:
resumo_tipo = df.groupby('TIPO').agg(
    municipios=('CITY', 'count'),
    idhm_mediano=('IDHM', 'median'),
    pib_pc_mediano=('GDP_CAPITA', 'median'),
    populacao_mediana=('POP_FINAL', 'median'),
    densidade_mediana=('DENSIDADE', 'median'),
).round(2)

resumo_tipo

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.boxplot(data=df, x='TIPO', y='IDHM', ax=axes[0])
axes[0].set_title('IDHM: capitais x interior')
axes[0].set_xlabel('Tipo de municipio')
axes[0].set_ylabel('IDHM')

base_gdp = df[(df['GDP_CAPITA'] > 0) & df['TIPO'].notna()].copy()
sns.boxplot(data=base_gdp, x='TIPO', y='GDP_CAPITA', ax=axes[1])
axes[1].set_yscale('log')
axes[1].set_title('PIB per capita: capitais x interior')
axes[1].set_xlabel('Tipo de municipio')
axes[1].set_ylabel('PIB per capita em escala log')

plt.tight_layout()
plt.show()

**Insight:** capitais tendem a concentrar populacao, servicos e infraestrutura. Ainda assim, o interior possui grande diversidade, incluindo municipios de alto desempenho e municipios vulneraveis.

### 6.4 PIB per capita explica IDHM?

In [ ]:
base_relacao = df[(df['GDP_CAPITA'] > 0) & df['IDHM'].notna() & df['REGIAO'].notna()].copy()

fig, ax = plt.subplots(figsize=(11, 6))
sns.scatterplot(
    data=base_relacao,
    x='GDP_CAPITA',
    y='IDHM',
    hue='REGIAO',
    size='POP_FINAL',
    sizes=(15, 220),
    alpha=0.6,
    ax=ax,
)
ax.set_xscale('log')
ax.set_title('Relacao entre PIB per capita e IDHM')
ax.set_xlabel('PIB per capita em escala log')
ax.set_ylabel('IDHM')
ax.legend(title='Regiao / populacao', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

correlacao = base_relacao[['GDP_CAPITA', 'IDHM']].corr().iloc[0, 1]
print(f'Correlacao entre PIB per capita e IDHM: {correlacao:.3f}')

**Insight:** renda municipal se relaciona positivamente com desenvolvimento humano, mas nao conta a historia inteira. Municipios com renda parecida podem ter IDHM diferente, indicando que politicas publicas, educacao, saude e estrutura urbana tambem importam.

### 6.5 Quais variaveis caminham juntas?

In [ ]:
corr_cols = ['IDHM', 'GDP_CAPITA', 'POP_FINAL', 'AREA', 'DENSIDADE', 'IBGE_PLANTED_AREA']
corr = df[corr_cols].replace([np.inf, -np.inf], np.nan).corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='vlag', center=0, fmt='.2f', ax=ax)
ax.set_title('Mapa de correlacao das variaveis selecionadas')
plt.tight_layout()
plt.show()

## 7. Storytelling com dados

O Brasil municipal nao e uniforme. A analise mostra um pais formado por poucos centros muito populosos e milhares de municipios pequenos, com realidades economicas e sociais diferentes. O IDHM ajuda a enxergar essa desigualdade porque resume dimensoes humanas que nao aparecem quando olhamos apenas para populacao ou PIB per capita.

A primeira parte da historia e a **concentracao**: populacao e atividade economica nao se distribuem igualmente. A segunda e a **desigualdade regional**: as regioes apresentam medianas e dispersoes diferentes de IDHM. A terceira e a **complexidade**: renda per capita contribui para o desenvolvimento, mas nao determina sozinha o resultado social.

In [ ]:
top_idhm = df[['CITY', 'STATE', 'REGIAO', 'IDHM', 'GDP_CAPITA', 'POP_FINAL']].dropna(subset=['IDHM']).sort_values('IDHM', ascending=False).head(10)
baixo_idhm_populoso = (
    df[['CITY', 'STATE', 'REGIAO', 'IDHM', 'GDP_CAPITA', 'POP_FINAL']]
    .dropna(subset=['IDHM', 'POP_FINAL'])
    .query('POP_FINAL >= 50000')
    .sort_values('IDHM')
    .head(10)
)

display(top_idhm)
display(baixo_idhm_populoso)

A comparacao entre municipios com maior IDHM e municipios populosos com baixo IDHM mostra que a analise exploratoria nao serve apenas para descrever o passado. Ela tambem aponta onde novas perguntas devem ser feitas: quais servicos faltam, quais politicas funcionam e onde uma intervencao pode gerar maior impacto social.

## 8. Conclusoes

1. O dataset e adequado para uma EDA introdutoria sobre desenvolvimento municipal, pois combina indicadores sociais, economicos, demograficos e territoriais.
2. IDHM, PIB per capita, populacao e densidade apresentam comportamentos diferentes; por isso, a interpretacao precisa cruzar mais de uma variavel.
3. As regioes brasileiras mostram diferencas relevantes de desenvolvimento, mas tambem existe desigualdade dentro de cada regiao.
4. Capitais tendem a apresentar melhores indicadores medianos, mas o interior nao deve ser analisado como um grupo unico e homogeneo.
5. PIB per capita tem relacao positiva com IDHM, mas desenvolvimento humano tambem depende de fatores sociais e institucionais.

### Limitacoes

- A EDA identifica padroes, mas nao prova causalidade.
- Algumas variaveis podem ter anos de referencia diferentes.
- Novas fontes, como educacao, saude, saneamento e seguranca, poderiam aprofundar a analise.

### Proximos passos

- Criar indicadores compostos de vulnerabilidade municipal.
- Comparar municipios semelhantes dentro da mesma regiao.
- Construir modelos preditivos simples para estimar IDHM a partir de variaveis socioeconomicas.